In [3]:
import pandas as pd

# carga de datos
ruta_absoluta_prod =  r'C:/Users/armil/Desktop/Begoña/data d4bs/CASOS/CASO FINAL 2/00 Datasets/productos_final.csv'
ruta_absoluta = r'C:/Users/armil/Desktop/Begoña/data d4bs/CASOS/CASO FINAL 2/00 Datasets/clientes_final.csv'

productos = pd.read_csv(ruta_absoluta_prod, parse_dates=['Fecha'])
clientes = pd.read_csv(ruta_absoluta, parse_dates=['Fecha_Nacimiento'])

# Cambio tipo fecha producto
productos['Fecha'] = pd.to_datetime(productos['Fecha'])

# Crear variable total venta
productos['Total_Venta'] = (productos['Cantidad'] * productos['Precio_Unitario'])

# Imputar tamano hogar por moda
moda_tamano_hogar = clientes.Tamano_Hogar.mode()[0]
clientes['Tamano_Hogar'] = clientes['Tamano_Hogar'].fillna(moda_tamano_hogar)

# imputar nivel educativo y ocupacion por categoria 'desconocido'

clientes['Nivel_Educativo'] = clientes['Nivel_Educativo'].fillna('Desconocido')
clientes['Ocupacion'] = clientes['Ocupacion'].fillna('Desconocido')

# Cambio tipo fecha clientes
clientes['Fecha_Nacimiento'] = pd.to_datetime(clientes['Fecha_Nacimiento'])


# Calculo de recency
fecha_max = productos['Fecha'].max()
rfm = productos.groupby('ID_Cliente').agg(Recency = ('Fecha', lambda x: (fecha_max - x.max()).days))
# Calculo de frequency
rfm['Frequency']= productos.groupby('ID_Cliente')['ID_Transaccion'].nunique()
# Calculo de monetary
rfm['Monetary']= productos.groupby('ID_Cliente')['Total_Venta'].sum()

#Creamos puntuaciones
rfm['r']=pd.cut(rfm['Recency'], bins = [0,60, 120, 180, 240, rfm['Recency'].max()], labels = [5,4,3,2,1], include_lowest = True)
rfm['f']=pd.cut(rfm['Frequency'], bins = [1,2, 4, 6, 9, rfm['Frequency'].max()], labels = [1,2,3,4,5], include_lowest = True)
rfm['m']=pd.cut(rfm['Monetary'], bins = [0,30,70, 150, 300, rfm['Monetary'].max()], labels = [1,2,3,4,5])

rfm['RFM']=  rfm['r'].astype(str) + rfm['f'].astype(str) + rfm['m'].astype(str)

# Definimos segmento
def segmentar_clientes(fila):
    r, f, m = str(fila['r']), str(fila['f']), str(fila['m'])
    
    # Definir condiciones previamente para simplificar el código
    es_alto_r = r in ['4', '5']
    es_alto_f = f in ['4', '5']
    es_alto_m = m in ['4', '5']
    
    es_medio_r = r in ['3']
    es_medio_f = f in ['3']
    es_medio_m = m in ['3']
    
    es_bajo_r = r in ['1', '2']
    es_bajo_f = f in ['1', '2']
    es_bajo_m = m in ['1', '2']
    
    # Aplicar la lógica de los segmentos
    if es_alto_r and es_alto_f and es_alto_m:
        return 'Campeones'
    elif es_alto_r and es_alto_f and (es_alto_m or es_medio_m):
        return 'Clientes Leales'
    elif es_medio_r and es_alto_f and es_alto_m:
        return 'Clientes Rentables'
    elif es_bajo_r and es_medio_f and es_medio_m:
        return 'Leales Potenciales'
    elif es_bajo_r and es_bajo_f and es_alto_m:
        return 'Grandes Gastadores'
    elif es_alto_r and es_bajo_f and es_bajo_m:
        return 'Clientes Recientes'
    elif es_medio_r and es_medio_f and es_medio_m:
        return 'Prometedores'
    elif es_bajo_r and es_bajo_f and es_bajo_m:
        return 'En Riesgo'
    else:
        return 'Dudosos'

# Aplicar la función
rfm['Segmento'] = rfm.apply(segmentar_clientes, axis=1)

# crear columna mes transacción
productos['MesTransaccion'] = productos['Fecha'].dt.to_period('M')

# Añadir mes de cohorte
productos['MesCohorte'] = productos.groupby('ID_Cliente')['MesTransaccion'].transform('min')

# eliminar registros de cohortes no completas
resultados = productos[productos['MesCohorte'] <= '2023-07']

# transformamos las variables Mes_ a datetime
resultados = resultados.copy()
resultados['MesTransaccion'] =pd.to_datetime(resultados['MesTransaccion'].astype(str), format = '%Y-%m')
resultados['MesCohorte'] =pd.to_datetime(resultados['MesCohorte'].astype(str), format = '%Y-%m')

# Elimar registros que no entran en el mes de estudio
resultados_filtrados = resultados[resultados['MesTransaccion'] <= resultados['MesCohorte'] + pd.DateOffset(months = 5)]


# Crear la variable del mes desde el punto de vista del cliente
resultados_filtrados = resultados_filtrados.copy()
# paso1: Crear columnas temporales tipo Period[M]
mes_transaccion_period = resultados_filtrados['MesTransaccion'].dt.to_period('M')
mes_cohorte_period = resultados_filtrados['MesCohorte'].dt.to_period('M')
# paso2 : calcular la diferencia en mese de columnas temporales
diferencia_meses = mes_transaccion_period -  mes_cohorte_period
# paso3: crear la nueva columna MesCliente con la diferencia calculada
resultados_filtrados.loc[:, 'MesCliente']  ='M'+diferencia_meses.apply(lambda x: str(x.n+1))

# Crear tabla pivotada a nivel de mes cliente
tabla_cohortes = pd.pivot_table(
    resultados_filtrados,
    index = 'MesCohorte',
    columns = 'MesCliente',
    values = 'ID_Cliente', 
    aggfunc = 'nunique').fillna(0).astype(int)

# Calcular tabla de retencion
tabla_retencion = tabla_cohortes.div(tabla_cohortes['M1'], axis = 0) *100
tabla_retencion = tabla_retencion.round(1)

# Calcular el CAC medio

resultados_filtrados = resultados_filtrados.merge(
    clientes[['ID_Cliente','CAC']], 
    on = 'ID_Cliente', 
    how = 'left'
)
# agregamos a nivel de cliente

cac_cliente = resultados_filtrados.groupby('ID_Cliente')['CAC'].first()

cac_medio = cac_cliente.mean()

# Calcular el LTV por cliente
ltv_cliente = resultados_filtrados.groupby('ID_Cliente')[['Total_Venta']].sum()
ltv_cliente.columns = ['ltv']


## tablon analitico

# Obtener sociodemográficos
sociodemo = clientes[['ID_Cliente','Genero','Nivel_Educativo','Ocupacion','Tamano_Hogar']]

tablon_analitico = resultados_filtrados.merge(
    rfm, 
    on = 'ID_Cliente',
    how = 'left'
)

# Unir variables sociodemograficas
tablon_analitico = tablon_analitico.merge(
    sociodemo, 
    on = 'ID_Cliente', 
    how = 'left'
)

tablon_analitico = tablon_analitico.merge(
    ltv_cliente,
    on = 'ID_Cliente',
    how = 'left'
)


## tablon a nivel cliente
tablon_cliente = tablon_analitico.groupby('ID_Cliente').agg({
    'MesCohorte':'first',
    'CAC':'first',
    'Recency':'first',
    'Frequency':'first',
    'Monetary':'first',
    'r':'first',
    'f':'first',
    'm':'first',
    'Segmento':'first',
    'Genero':'first',
    'Nivel_Educativo':'first',
    'Ocupacion':'first',
    'Tamano_Hogar':'first',
    'ltv':'first'
})



### tablon ltv_mes

ltv_mes = tablon_analitico.groupby(['ID_Cliente','MesCliente']).agg({'Total_Venta':'sum'}).reset_index()

ltv_agrupado = ltv_mes.groupby('MesCliente').agg(
    {'Total_Venta':'sum',
     'ID_Cliente':'nunique'}
)
# Calcular la base del estudio
ltv_agrupado['M1'] = ltv_agrupado.loc['M1','ID_Cliente']

ltv_agrupado['Ventas_por_M1']   = ltv_agrupado['Total_Venta'] / ltv_agrupado['M1']
ltv_agrupado = ltv_agrupado.reset_index()


# Exportar tablon final
tablon_analitico.to_csv('C:/Users/armil/Desktop/Begoña/data d4bs/CASOS/CASO FINAL 2/00 Datasets/tablon_analitico_clientes.csv', decimal = ',', index = False)

# Exportar tablon nivel cliente
tablon_cliente.to_csv('C:/Users/armil/Desktop/Begoña/data d4bs/CASOS/CASO FINAL 2/00 Datasets/tablon_cliente.csv', decimal = ',', index = False)


# Exportar tablon ltv_mes
ltv_agrupado.to_csv('C:/Users/armil/Desktop/Begoña/data d4bs/CASOS/CASO FINAL 2/00 Datasets/ltv_mes.csv', decimal = ',', index = False)


In [4]:
# Exportar tablon final
tablon_analitico.to_csv('C:/Users/armil/Desktop/Begoña/data d4bs/CASOS/CASO FINAL 2/00 Datasets/tablon_analitico_clientes.csv', decimal = ',', index = False)

In [5]:
# Exportar tablon nivel cliente
tablon_cliente.to_csv('C:/Users/armil/Desktop/Begoña/data d4bs/CASOS/CASO FINAL 2/00 Datasets/tablon_cliente.csv', decimal = ',', index = False)

In [7]:
# Exportar tablon ltv_mes
ltv_agrupado.to_csv('C:/Users/armil/Desktop/Begoña/data d4bs/CASOS/CASO FINAL 2/00 Datasets/ltv_mes.csv', decimal = ',', index = False)

In [8]:
tablon_analitico.info()

<class 'pandas.DataFrame'>
RangeIndex: 54405 entries, 0 to 54404
Data columns (total 26 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   ID_Cliente          54405 non-null  int64         
 1   Fecha               54405 non-null  datetime64[us]
 2   ID_Transaccion      54405 non-null  int64         
 3   Categoria_Producto  54405 non-null  str           
 4   Producto            54405 non-null  str           
 5   Coste_Unitario      54405 non-null  float64       
 6   Cantidad            54405 non-null  float64       
 7   Precio_Unitario     54405 non-null  float64       
 8   Total_Venta         54405 non-null  float64       
 9   MesTransaccion      54405 non-null  datetime64[us]
 10  MesCohorte          54405 non-null  datetime64[us]
 11  MesCliente          54405 non-null  str           
 12  CAC                 54405 non-null  float64       
 13  Recency             54405 non-null  int64         
 14  F

In [9]:
tablon_analitico.groupby('ID_Cliente').head()

,ID_Cliente,Fecha,ID_Transaccion,Categoria_Producto,Producto,Coste_Unitario,Cantidad,Precio_Unitario,Total_Venta,MesTransaccion,...,r,f,m,RFM,Segmento,Genero,Nivel_Educativo,Ocupacion,Tamano_Hogar,ltv
0,1,2023-01-21,3442,0H2,6OUVC,7.64,4.0,10.50,42.00,2023-01-01,...,1,1,2,112,En Riesgo,Hombre,Universitario,Empleado,4-5 personas,65.16
1,1,2023-01-21,3442,N8U,CEBU8,4.57,4.0,5.79,23.16,2023-01-01,...,1,1,2,112,En Riesgo,Hombre,Universitario,Empleado,4-5 personas,65.16
2,2,2023-03-24,14177,TVL,2SLS0,4.60,3.0,7.77,23.31,2023-03-01,...,2,1,2,212,En Riesgo,Hombre,Sin estudios,Empleado,1 persona,68.31
3,2,2023-06-19,30451,F9B,GZ6VU,9.04,3.0,15.00,45.00,2023-06-01,...,2,1,2,212,En Riesgo,Hombre,Sin estudios,Empleado,1 persona,68.31
4,3,2023-01-01,90,LPF,Y1M2E,3.37,4.0,4.08,16.32,2023-01-01,...,1,1,1,111,En Riesgo,Mujer,Bachillerato,Estudiante,4-5 personas,24.60
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54400,2494,2023-07-27,36500,FU5,3ZY0H,3.34,3.0,4.96,14.88,2023-07-01,...,3,1,1,311,Dudosos,Mujer,Bachillerato,Empleado,2-3 personas,14.88
54401,3527,2023-04-28,20425,FEW,WG7HI,4.11,3.0,5.18,15.54,2023-04-01,...,1,1,1,111,En Riesgo,Mujer,Universitario,Estudiante,2-3 personas,15.54
54402,5063,2023-03-13,11915,29A,PO1N6,3.00,10.0,4.96,49.60,2023-03-01,...,1,1,2,112,En Riesgo,Mujer,Universitario,Jubilado,2-3 personas,49.60
54403,17354,2023-06-29,32041,01F,RVG1H,1.47,3.0,1.68,5.04,2023-06-01,...,2,1,1,211,En Riesgo,Mujer,Bachillerato,Empleado,2-3 personas,5.04


In [10]:
# conteo de valores unicos por columna para cada cliente
valores_unicos_cliente = tablon_analitico.groupby('ID_Cliente').nunique()

# Identificar columnas con mas de un valor unico
valores_unicos_cliente.loc[:, (valores_unicos_cliente> 1).any()]

,Fecha,ID_Transaccion,Categoria_Producto,Producto,Coste_Unitario,Cantidad,Precio_Unitario,Total_Venta,MesTransaccion,MesCliente
ID_Cliente,,,,,,,,,,
1,1,1,2,2,2,1,2,2,1,1
2,2,2,2,2,2,1,2,2,2,2
3,1,1,2,3,3,2,2,2,1,1
4,2,2,5,5,5,2,5,5,2,2
5,5,5,2,2,5,5,4,5,4,4
...,...,...,...,...,...,...,...,...,...,...
22232,1,1,1,1,1,1,1,1,1,1
22233,1,1,1,1,1,1,1,1,1,1
22234,2,2,3,4,5,4,5,5,2,2


In [11]:
tablon_cliente = tablon_analitico.groupby('ID_Cliente').agg({
    'MesCohorte':'first',
    'CAC':'first',
    'Recency':'first',
    'Frequency':'first',
    'Monetary':'first',
    'r':'first',
    'f':'first',
    'm':'first',
    'Segmento':'first',
    'Genero':'first',
    'Nivel_Educativo':'first',
    'Ocupacion':'first',
    'Tamano_Hogar':'first',
    'ltv':'first'
})
tablon_cliente

,MesCohorte,CAC,Recency,Frequency,Monetary,r,f,m,Segmento,Genero,Nivel_Educativo,Ocupacion,Tamano_Hogar,ltv
ID_Cliente,,,,,,,,,,,,,,
1,2023-01-01,184.31,344,1,65.16,1,1,2,En Riesgo,Hombre,Universitario,Empleado,4-5 personas,65.16
2,2023-03-01,31.60,195,2,68.31,2,1,2,En Riesgo,Hombre,Sin estudios,Empleado,1 persona,68.31
3,2023-01-01,29.33,364,1,24.60,1,1,1,En Riesgo,Mujer,Bachillerato,Estudiante,4-5 personas,24.60
4,2023-07-01,63.67,52,2,139.85,5,1,3,Dudosos,Hombre,Bachillerato,Empleado,2-3 personas,139.85
5,2023-02-01,11.13,179,5,143.81,3,3,3,Prometedores,Mujer,Universitario,Estudiante,2-3 personas,143.81
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22232,2023-06-01,48.37,193,1,21.00,2,1,1,En Riesgo,Hombre,Universitario,Autónomo,2-3 personas,21.00
22233,2023-07-01,20.30,179,1,23.76,3,1,1,Dudosos,Mujer,Universitario,Empleado,2-3 personas,23.76
22234,2023-05-01,81.66,142,2,221.28,3,1,4,Dudosos,Mujer,Máster,Jubilado,2-3 personas,221.28


In [12]:
tablon_cliente.CAC.mean()

np.float64(69.44205818890514)

In [13]:
tablon_analitico.Segmento.value_counts()

Segmento
Dudosos               26874
En Riesgo             10108
Grandes Gastadores     8165
Campeones              6424
Clientes Recientes     1177
Clientes Leales         526
Clientes Rentables      501
Leales Potenciales      493
Prometedores            137
Name: count, dtype: int64

In [14]:
ltv_mes = tablon_analitico.groupby(['ID_Cliente','MesCliente']).agg({'Total_Venta':'sum'}).reset_index()

ltv_agrupado = ltv_mes.groupby('MesCliente').agg(
    {'Total_Venta':'sum',
     'ID_Cliente':'nunique'}
)
# Calcular la base del estudio
ltv_agrupado['M1'] = ltv_agrupado.loc['M1','ID_Cliente']

ltv_agrupado['Ventas_por_M1']   = ltv_agrupado['Total_Venta'] / ltv_agrupado['M1']
ltv_agrupado.reset_index()

,MesCliente,Total_Venta,ID_Cliente,M1,Ventas_por_M1
0,M1,885178.76,14367,14367,61.611941
1,M2,193674.15,3003,14367,13.480487
2,M3,140703.96,2390,14367,9.793552
3,M4,96188.18,1790,14367,6.695078
4,M5,80492.23,1495,14367,5.602577
5,M6,80064.04,1417,14367,5.572774


In [15]:
ltv_agrupado['Ventas_por_M1'].sum()

np.float64(102.75640843599915)

In [16]:
import sys
print(sys.executable)

C:\Users\armil\anaconda3\envs\CASO_2CA\python.exe
